In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
import pandas as pd
import time
import os

service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service)

all_jobs = []

def safe_extract(card, by_type, selector, attribute=None):
    try:
        element = card.find_element(by_type, selector)
        return element.get_attribute(attribute) if attribute else element.text
    except:
        return None

def scrape_current_page(page_num):
    job_cards = driver.find_elements(By.CLASS_NAME, "srp-jobtuple-wrapper")
    page_jobs = []
    
    for card in job_cards:
        job_id = card.get_attribute("data-job-id")
        title = safe_extract(card, By.CLASS_NAME, "title", "title")
        company = safe_extract(card, By.CLASS_NAME, "comp-name")
        rating = safe_extract(card, By.CLASS_NAME, "main-2")
        experience = safe_extract(card, By.CLASS_NAME, "expwdth", "title")
        location = safe_extract(card, By.CLASS_NAME, "locWdth", "title")
        salary = safe_extract(card, By.CLASS_NAME, "sal-wrap")
        description = safe_extract(card, By.CLASS_NAME, "job-desc")
        posted_date = safe_extract(card, By.CLASS_NAME, "job-post-day")
        
        try:
            tag_elements = card.find_elements(By.CSS_SELECTOR, "ul.tags-gt li")
            skill_tags = [tag.text for tag in tag_elements]
        except:
            skill_tags = []
        
        if title:
            page_jobs.append({
                "job_id": job_id,
                "title": title,
                "company": company,
                "rating": rating,
                "experience": experience,
                "location": location,
                "salary": salary,
                "posted_date": posted_date,
                "description": description,
                "skill_tags": skill_tags,
                "page": page_num
            })
    
    return page_jobs

driver.get("https://www.naukri.com/data-analyst-jobs-in-india?k=data+analyst&l=india")
time.sleep(5)

for page_num in range(1, 16):
    print(f"Scraping page {page_num}...")
    jobs = scrape_current_page(page_num)
    all_jobs.extend(jobs)
    print(f"Got {len(jobs)} jobs, total so far: {len(all_jobs)}")
    
    try:
        next_button = driver.find_element(By.XPATH, "//a[contains(@class,'styles_btn-secondary') and .//span[text()='Next']]")
        driver.execute_script("arguments[0].click();", next_button)
        time.sleep(5)
    except Exception as e:
        print("Next button nahi mila, ruk rahe hain:", e)
        break

driver.quit()

project_folder = r"D:\SUBJECTS\Data_Analyst\AnalystProjects\datanalystproject"
os.makedirs(project_folder, exist_ok=True)

df = pd.DataFrame(all_jobs)
save_path = os.path.join(project_folder, "raw_jobs_final.csv")
df.to_csv(save_path, index=False)

print("\n✅ Final Saved!")
print("Total jobs collected:", len(df))
print("Unique job_ids:", df["job_id"].nunique())
print("\nMissing values per column:")
print(df.isnull().sum())

print("\n--- Sample Preview ---")
print(df[["title", "company", "rating", "salary", "posted_date"]].head(10))

Scraping page 1...
Got 20 jobs, total so far: 20
Scraping page 2...
Got 20 jobs, total so far: 40
Scraping page 3...
Got 20 jobs, total so far: 60
Scraping page 4...
Got 20 jobs, total so far: 80
Scraping page 5...
Got 20 jobs, total so far: 100
Scraping page 6...
Got 20 jobs, total so far: 120
Scraping page 7...
Got 20 jobs, total so far: 140
Scraping page 8...
Got 20 jobs, total so far: 160
Scraping page 9...
Got 20 jobs, total so far: 180
Scraping page 10...
Got 20 jobs, total so far: 200
Scraping page 11...
Got 20 jobs, total so far: 220
Scraping page 12...
Got 20 jobs, total so far: 240
Scraping page 13...
Got 20 jobs, total so far: 260
Scraping page 14...
Got 20 jobs, total so far: 280
Scraping page 15...
Got 19 jobs, total so far: 299

✅ Final Saved!
Total jobs collected: 299
Unique job_ids: 299

Missing values per column:
job_id           0
title            0
company          0
rating         108
experience       4
location         2
salary         233
posted_date      0
descri